# CelebA-HQ — Stage 5 metadata-only loader/split binding

Run with **Accelerator = None**. Attach the DIRE input Dataset, the code Dataset version containing `dire_modal/`, and the Stage 3 v2 split artifact (or keep its JSON under `/kaggle/working`). This cell validates paths/counts/fingerprint only: it does not open image contents, create tensors, use GPU, or train. The complete metadata scan can take time; the latest loader package prints progress during path indexing and fingerprinting. Older attached package versions still work, but may remain quiet until the scan completes.

The output report is written only to `/kaggle/working/celebahq-data-binding-report.json`; the mounted input remains read-only. Do not index/iterate the returned datasets in this stage.

In [ ]:
from pathlib import Path
import sys

INPUT_ROOT = Path('/kaggle/input')
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
SPLIT_ARTIFACT = Path('/kaggle/working/celebahq-validation-split-v2-81dcda99aeb9.json')

# Kaggle mounts attached Datasets but does not add their Python packages to sys.path.
# Search only mount roots (and optional src/), never recurse through the image tree.
mount_roots = []
for mount in INPUT_ROOT.iterdir():
    if mount.name == 'datasets' and mount.is_dir():
        for owner in mount.iterdir():
            if owner.is_dir():
                mount_roots.extend(dataset for dataset in owner.iterdir() if dataset.is_dir())
    elif mount.is_dir():
        mount_roots.append(mount)
code_roots = [
    root for mount in mount_roots for root in (mount, mount / 'src')
    if (root / 'dire_modal' / '__init__.py').is_file()
    and (root / 'dire_modal' / 'kaggle_celebahq_data.py').is_file()
]
if len(code_roots) != 1:
    listing = {str(mount): [child.name for child in mount.iterdir()][:12] for mount in mount_roots}
    raise FileNotFoundError(f'Expected one attached code package containing dire_modal; found {code_roots}. Mount listing: {listing}')
CODE_DATASET_ROOT = code_roots[0]
if not DIRE_ROOT.is_dir():
    raise FileNotFoundError(f'Missing DIRE input root: {DIRE_ROOT}')
if not SPLIT_ARTIFACT.is_file():
    raise FileNotFoundError(f'Missing Stage 3 v2 split artifact: {SPLIT_ARTIFACT}')

sys.path.insert(0, str(CODE_DATASET_ROOT))
from dire_modal.kaggle_celebahq_data import build_celebahq_datasets

print('Code package root added to sys.path:', CODE_DATASET_ROOT)
print('DIRE root:', DIRE_ROOT)
print('Split artifact:', SPLIT_ARTIFACT)

In [ ]:
from collections import Counter
import inspect
import json

def progress(message: str) -> None:
    print('[binding]', message, flush=True)

builder_kwargs = {}
if 'progress_callback' in inspect.signature(build_celebahq_datasets).parameters:
    builder_kwargs['progress_callback'] = progress
else:
    print('Attached code package lacks progress callback; scan still runs without intermediate progress.')

bundle = build_celebahq_datasets(DIRE_ROOT, SPLIT_ARTIFACT, **builder_kwargs)

train_counts = Counter(record.label for record in bundle.train.records)
validation_counts = Counter(record.label for record in bundle.validation.records)
test_counts = {name: len(dataset) for name, dataset in bundle.test_by_generator.items()}

expected_train = {0: 25_200, 1: 36_000}
expected_validation = {0: 2_800, 1: 4_000}
# Inventory counts previously reported for this mounted CelebA-HQ test tree.
expected_test = {
    'real': 1_000, 'dalle2': 500, 'if': 1_000,
    'midjourney': 100, 'stargan': 1_000, 'sdv2': 1_000,
}

assert dict(train_counts) == expected_train, (dict(train_counts), expected_train)
assert dict(validation_counts) == expected_validation, (dict(validation_counts), expected_validation)
assert test_counts == expected_test, (test_counts, expected_test)
assert bundle.validation_id_list_sha256 == (
    'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
)

report = {
    'status': 'pass',
    'protocol': 'celebahq-content-disjoint-exact-count-hash-rank-v2',
    'source_fingerprint': bundle.source_fingerprint,
    'validation_id_list_sha256': bundle.validation_id_list_sha256,
    'train_count_by_label': {str(k): v for k, v in sorted(train_counts.items())},
    'validation_count_by_label': {str(k): v for k, v in sorted(validation_counts.items())},
    'test_count_by_generator': test_counts,
    'image_contents_read': False,
    'input_write_attempted': False,
    'gpu_workload_started': False,
    'training_started': False,
}
report_path = Path('/kaggle/working/celebahq-data-binding-report.json')
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved report:', report_path)

In [ ]:
# Run this cell after the binding cell stops on the metadata-fingerprint mismatch.
# It verifies current file bytes against Stage 2's saved SHA-256 values. It does
# not bypass the loader guard, change split IDs, decode images, use a GPU, or
# write under either mounted Dataset. Only the JSON report is written to working.
import hashlib
import inspect
import json
import re
import sqlite3

EXPECTED_STAGE2_FINGERPRINT = '81dcda99aeb997b25b82e115191e4babfdd2957110706584f10dbd827991e6da'
EXPECTED_RECORDS = 72_600
STAGE2_DB = Path(
    '/kaggle/input/datasets/j4cknguyn/dire-celeba-hq-stage-2-artifacts-v1/'
    'celebahq-integrity-81dcda99aeb997b25b82.sqlite3'
)
SHA_REPORT = Path('/kaggle/working/celebahq-stage2-sha256-compare-report.json')

if 'DIRE_ROOT' not in globals() or 'CODE_DATASET_ROOT' not in globals():
    raise RuntimeError('Run the setup cell first; this cell reuses its selected code package and DIRE_ROOT.')
if not DIRE_ROOT.is_dir():
    raise FileNotFoundError(f'Missing DIRE input root: {DIRE_ROOT}')
if not STAGE2_DB.is_file():
    raise FileNotFoundError(f'Missing attached Stage 2 SQLite artifact: {STAGE2_DB}')
if DIRE_ROOT.resolve() in STAGE2_DB.resolve().parents:
    raise ValueError('Stage 2 database unexpectedly resides inside the read-only DIRE input tree.')
if not str(SHA_REPORT.resolve()).startswith('/kaggle/working/'):
    raise ValueError(f'Report path must remain outside mounted inputs: {SHA_REPORT}')

from dire_modal.kaggle_celebahq_tree import scan_celebahq_tree
scanner_path = Path(inspect.getsourcefile(scan_celebahq_tree)).resolve()
if CODE_DATASET_ROOT.resolve() not in scanner_path.parents:
    raise RuntimeError(
        f'Scanner was imported from outside the selected code Dataset: {scanner_path}; '
        f'selected package root={CODE_DATASET_ROOT}'
    )
print('Using the already-selected code package:', CODE_DATASET_ROOT)
print('Scanner module:', scanner_path)
print('DIRE root:', DIRE_ROOT)
print('Stage 2 SQLite (opened read-only):', STAGE2_DB)

# Read only: mode=ro prevents SQLite from creating or modifying a journal/database.
stage2_db = sqlite3.connect(STAGE2_DB.resolve().as_uri() + '?mode=ro', uri=True)
try:
    quick_check = stage2_db.execute('PRAGMA quick_check').fetchone()[0]
    if quick_check != 'ok':
        raise RuntimeError(f'Stage 2 SQLite integrity check failed: {quick_check}')
    meta = dict(stage2_db.execute('SELECT key, value FROM meta'))
    if meta.get('root') != str(DIRE_ROOT.resolve()):
        raise RuntimeError(
            f'Stage 2 DB root mismatch: {meta.get("root")!r} != {str(DIRE_ROOT.resolve())!r}'
        )
    if meta.get('source_fingerprint') != EXPECTED_STAGE2_FINGERPRINT:
        raise RuntimeError(
            'Stage 2 DB fingerprint mismatch: '
            f'{meta.get("source_fingerprint")!r} != {EXPECTED_STAGE2_FINGERPRINT!r}'
        )
    if int(meta.get('record_count', '-1')) != EXPECTED_RECORDS:
        raise RuntimeError(f'Unexpected Stage 2 DB record_count: {meta.get("record_count")!r}')
    db_rows = stage2_db.execute(
        '''SELECT relative_path, split, generator, label, member_locator,
                  locator_inferred, size_bytes, mtime_ns, sha256, hash_status
           FROM files ORDER BY relative_path'''
    ).fetchall()
finally:
    stage2_db.close()

if len(db_rows) != EXPECTED_RECORDS:
    raise RuntimeError(f'Stage 2 files table has {len(db_rows):,} rows; expected {EXPECTED_RECORDS:,}.')
stage2_by_path = {
    row[0]: {
        'split': row[1], 'generator': row[2], 'label': int(row[3]),
        'member_locator': row[4], 'locator_inferred': bool(row[5]),
        'size_bytes': int(row[6]), 'mtime_ns': int(row[7]),
        'sha256': row[8], 'hash_status': row[9],
    }
    for row in db_rows
}
if len(stage2_by_path) != EXPECTED_RECORDS:
    raise RuntimeError('Stage 2 database contains duplicate relative paths.')


def scan_progress(message: str) -> None:
    print('[sha-audit]', message, flush=True)

current_records = scan_celebahq_tree(DIRE_ROOT, progress_callback=scan_progress)
current_by_path = {record.relative_path: record for record in current_records}
if len(current_by_path) != len(current_records):
    raise RuntimeError('Current tree scanner returned duplicate relative paths.')
missing_paths = sorted(set(stage2_by_path) - set(current_by_path))
extra_paths = sorted(set(current_by_path) - set(stage2_by_path))
identity_mismatches = []
size_mismatches = []
mtime_mismatches = 0
invalid_stage2_hashes = []

for relative_path in sorted(set(stage2_by_path) & set(current_by_path)):
    old = stage2_by_path[relative_path]
    new = current_by_path[relative_path]
    observed_identity = (new.split, new.generator, int(new.label), new.member_locator, bool(new.locator_inferred))
    expected_identity = (old['split'], old['generator'], old['label'], old['member_locator'], old['locator_inferred'])
    if observed_identity != expected_identity:
        identity_mismatches.append(relative_path)
    if int(new.size_bytes) != old['size_bytes']:
        size_mismatches.append(relative_path)
    if int(new.mtime_ns) != old['mtime_ns']:
        mtime_mismatches += 1
    if old['hash_status'] != 'pass' or not isinstance(old['sha256'], str) or not re.fullmatch(r'[0-9a-fA-F]{64}', old['sha256']):
        invalid_stage2_hashes.append(relative_path)

precheck = {
    'stage2_records': len(stage2_by_path),
    'current_records': len(current_by_path),
    'missing_paths': len(missing_paths),
    'extra_paths': len(extra_paths),
    'identity_mismatches': len(identity_mismatches),
    'size_mismatches': len(size_mismatches),
    'mtime_ns_mismatches': mtime_mismatches,
    'invalid_or_failed_stage2_hashes': len(invalid_stage2_hashes),
}
print('Metadata precheck:', json.dumps(precheck, sort_keys=True))
if any((missing_paths, extra_paths, identity_mismatches, size_mismatches, invalid_stage2_hashes)):
    report = {
        'status': 'fail_precheck',
        'stage2_fingerprint': EXPECTED_STAGE2_FINGERPRINT,
        'stage2_sqlite': str(STAGE2_DB),
        'dire_root': str(DIRE_ROOT.resolve()),
        'precheck': precheck,
        'samples': {
            'missing_paths': missing_paths[:10], 'extra_paths': extra_paths[:10],
            'identity_mismatches': identity_mismatches[:10],
            'size_mismatches': size_mismatches[:10],
            'invalid_or_failed_stage2_hashes': invalid_stage2_hashes[:10],
        },
        'image_bytes_read': False,
        'image_decode_performed': False,
        'gpu_workload_started': False,
        'mounted_input_write_attempted': False,
    }
    SHA_REPORT.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(report, indent=2))
    raise RuntimeError(f'Precheck failed; no image hashing was done. Report: {SHA_REPORT}')

hash_matches = 0
hash_mismatches = []
hash_errors = []
unstable_during_read = []
for index, relative_path in enumerate(sorted(current_by_path), start=1):
    record = current_by_path[relative_path]
    path = Path(record.path)
    try:
        before = path.stat()
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        after = path.stat()
        if (before.st_size, before.st_mtime_ns, before.st_ino) != (after.st_size, after.st_mtime_ns, after.st_ino):
            unstable_during_read.append(relative_path)
            continue
        expected_sha256 = stage2_by_path[relative_path]['sha256'].lower()
        actual_sha256 = digest.hexdigest()
        if actual_sha256 == expected_sha256:
            hash_matches += 1
        else:
            hash_mismatches.append({
                'path': relative_path,
                'stage2_sha256': expected_sha256,
                'current_sha256': actual_sha256,
            })
    except Exception as exc:
        hash_errors.append({'path': relative_path, 'error': f'{type(exc).__name__}: {exc}'})
    if index % 1000 == 0 or index == len(current_by_path):
        print(
            f'Hashed {index:,}/{len(current_by_path):,}; '
            f'matches={hash_matches:,}; mismatches={len(hash_mismatches):,}; '
            f'errors={len(hash_errors):,}; unstable={len(unstable_during_read):,}',
            flush=True,
        )

status = 'pass' if (
    hash_matches == EXPECTED_RECORDS
    and not hash_mismatches and not hash_errors and not unstable_during_read
) else 'fail'
report = {
    'status': status,
    'comparison': 'current mounted file SHA-256 vs Stage 2 SQLite SHA-256',
    'stage2_fingerprint': EXPECTED_STAGE2_FINGERPRINT,
    'stage2_sqlite': str(STAGE2_DB),
    'dire_root': str(DIRE_ROOT.resolve()),
    'precheck': precheck,
    'records_attempted': hash_matches + len(hash_mismatches) + len(hash_errors) + len(unstable_during_read),
    'hashes_compared': hash_matches + len(hash_mismatches),
    'hash_matches': hash_matches,
    'hash_mismatches': len(hash_mismatches),
    'hash_errors': len(hash_errors),
    'unstable_during_read': len(unstable_during_read),
    'samples': {
        'hash_mismatches': hash_mismatches[:10],
        'hash_errors': hash_errors[:10],
        'unstable_during_read': unstable_during_read[:10],
    },
    'image_bytes_read_for_sha256': True,
    'image_decode_performed': False,
    'gpu_workload_started': False,
    'training_started': False,
    'mounted_input_write_attempted': False,
    'report_written_to': str(SHA_REPORT),
    'interpretation': (
        'PASS means current bytes match all Stage 2 hashes; it does not prove original archive provenance '
        'and does not bypass the metadata fingerprint guard. A separately reviewed, versioned binding '
        'decision is still required.' if status == 'pass' else
        'FAIL means byte identity was not established; do not reuse the split artifact.'
    ),
}
SHA_REPORT.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved read-only comparison report:', SHA_REPORT)
if status != 'pass':
    raise RuntimeError(f'Stage 2/current SHA-256 comparison failed; see {SHA_REPORT}')
